# Dissipative case — the 2.5PN inspiral survey

24 runs in one configuration: 14 at six radial periods and 10 at ten. `P1`–`P14` are the published rows, `D1`–`D10` extend the analysis. Only the initial data and `n_orbits` change between runs, which makes the survey a one-factor study.

The trained runs of the paper are in `models/dissipative/`; new runs are written to `runs/diss_all/`.

In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DISS_DIR = str(REPO / "models" / "dissipative")   # the trained runs of the paper
RUNS_DIR = str(REPO / "runs" / "diss_all")        # where NEW runs are written

import gravinns.constants as const
from gravinns.utils import get_device, set_global_seed
from gravinns.experiments.dissipative import (COMMON, IC_SET, IC_BY_TAG, TAGS,
                                              show_ic_set, run_case, paper_table,
                                              show_paper_table, latex_paper_table,
                                              energy_diagnostics, run_dir, load_model)
from gravinns.plotting import (load_plot_record, list_energy_snapshots,
                               replot_training_figure, plot_energy_evolution)

set_global_seed(42)
device = get_device(verbose=True)
const.print_normalization_summary()

## 1. The shared configuration

Everything below is identical across all 24 runs (`nu` and `pr0_factor` are per run and live in the run list). Edit anything here, then pass `common=COMMON` to `run_case`.

In [ ]:
# Shared by all 24 runs. nu and pr0_factor are per-run and are NOT here.
COMMON = dict(
    dissipative=True,
    no_25pn_data=True,          # PHYSICS-ONLY (audited at first backward pass)
    l2_target=1e-3, gap_factor=10.0,
    w_secular=50.0,
    w_L=1.0, w_anchor=10.0,
    n_harmonic=0,
    n_march=10, march_fraction=0.4,
    use_causal_training=False,
    transfer_from_2pn=True,
    pretrain_epochs=10_000, pretrain_lr=1e-3,
    curriculum_start=1_000, curriculum_epochs=20_000,
    n_fourier=12, fourier_sigma=3.0,
    n_fourier_hi=20, fourier_sigma_hi=8.0,
    hidden=128, depth=5, lr=1e-3,
    n_colloc=6000,
    n_epochs=250_000,
    convergence_window=20, convergence_cv=0.5,
    compute_l2_time=False,
    plot_every=5_000, log_every=2_000, checkpoint_every=10_000,
    record_every_loss=500, record_every_l2=500, record_every_energy=5_000,
)

print(f"{len(COMMON)} shared settings; n_epochs = {COMMON['n_epochs']:,}")

## 2. The 24 runs

`show_ic_set` prints the list with the published L2 where known and marks which runs already have a trained folder. The full annotated table (e0, N_rev, e0*N_rev, r_p/r_g, gap and the reasoning behind each new run) is in the docstring of `gravinns.experiments.dissipative`.

In [ ]:
ic = show_ic_set(base_dir=DISS_DIR)

## 3. Train one case

Select the run by its tag (`'D1'`, `'P4'`, …). Any keyword given to `run_case` overrides `COMMON` for that run, e.g. `run_case(TAG, common=COMMON, n_epochs=50_000)` for a quick test.

In [ ]:
TAG = "D1"          # <-- the case to run; see the list above

diag, best_l2 = run_case(TAG, common=COMMON, base_dir=RUNS_DIR, dev=device)

## 4. The paper's table

`paper_table` rebuilds one row per trained run directly from its folder (config, `analysis_summary.json`, `analysis.npz` and `angle_best.pth`) and saves it as `paper_table.csv` / `.json` inside that folder, so it never has to be recomputed.

Columns: `N_orb`, `nu`, `p0_hat`, `r0` (in R0 = 200 km), `e0`, `N_rev`, `r_p/r_g`, `gap` (the 2PN↔2.5PN signal), `L2`, and the energy check `dH [%]` with the fraction of rising steps.

In [ ]:
t = paper_table(DISS_DIR)      # also writes paper_table.csv / .json there
show_paper_table(table=t)

In [ ]:
# the same rows as LaTeX
print(latex_paper_table(table=t))

## 5. Energy diagnostics of one run

A true 2.5PN inspiral radiates, so H(φ) must decrease monotonically: rising steps are the unphysical fraction. `energy_diagnostics` recomputes H(φ) from the trained network and from the 2.5PN reference on the same grid.

In [ ]:
d = energy_diagnostics(run_dir("D1", DISS_DIR))
print(f"network  dH = {d['dH_pct']:+.1f}%   rising steps = {100*d['frac_up']:.1f}%")
print(f"2.5PN ref dH = {d['dH_ref_pct']:+.1f}%")

plt.figure(figsize=(7, 4))
plt.plot(d["phi"], d["H"], label="H(φ) — PINN")
plt.xlabel("φ [rad]"); plt.ylabel("H"); plt.legend(); plt.grid(alpha=.3); plt.show()